In [0]:
import requests
import pandas as pd

# Replace with your API endpoint
api_url = "https://services.nvd.nist.gov/rest/json/cves/2.0"
response = requests.get(api_url)
data = response.json()

# Extract the vulnerabilities list
vulns = data.get('vulnerabilities', [])

# Flatten key fields from each CVE entry
flat_records = []
for item in vulns:
    cve = item.get('cve', {})
    cve_id = cve.get('id')
    published = cve.get('published')
    last_modified = cve.get('lastModified')
    # Get English description
    descriptions = cve.get('descriptions', [])
    description = next((d['value'] for d in descriptions if d['lang'] == 'en'), None)
    # Get CVSS v2 base score if available
    metrics = cve.get('metrics', {})
    cvss_v2 = metrics.get('cvssMetricV2', [{}])[0].get('cvssData', {})
    base_score = cvss_v2.get('baseScore')
    severity = metrics.get('cvssMetricV2', [{}])[0].get('baseSeverity')
    flat_records.append({
        'cve_id': cve_id,
        'published': published,
        'last_modified': last_modified,
        'description': description,
        'base_score': base_score,
        'severity': severity
    })

# Create pandas DataFrame and convert to Spark DataFrame
pdf = pd.DataFrame(flat_records)
df = spark.createDataFrame(pdf)
display(df)